# APASA workbook inspection and candidate crosswalk
Review stage only — no cleaned observations, seasonal sums, conversions, or target comparisons.

Run cells in order. Inputs are opened read-only in intent and are never saved. SHA-256 checks verify unchanged bytes. Source labels, formulas, cached results, codes and blanks remain available in the cell audit. Candidate links are provisional interpretations, not verified equivalences.

The supplied directory has multiple NISR workbooks, not one consolidated history. Coverage is reported from evidence inside each sheet, with filename years identified separately. The register has six recurring indicator definitions and thirty seasonal records; stable local IDs are analytical identifiers, not official PSTA5 codes. Register baseline/target fields are kept in a separate register audit.


In [1]:
from pathlib import Path
import sys, json, csv, re, hashlib, html
from collections import Counter, defaultdict
import openpyxl
if Path('/tmp/apasa_workbook_deps').exists():
    sys.path.insert(0, '/tmp/apasa_workbook_deps')
import xlrd
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').is_dir():
    ROOT = Path('/home/kali/Documents/Projects/apasa-dashboard')
OUT = ROOT / 'outputs/workbook_review_2026_10_28'
OUT.mkdir(parents=True, exist_ok=True)
inputs = sorted(p for p in (ROOT/'data').iterdir() if p.suffix.lower() in ('.xls','.xlsx','.xlsm'))
hash_file = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
before = {str(p.relative_to(ROOT)): hash_file(p) for p in inputs}
print('Excel inputs:', len(inputs))
for p in inputs: print(p.name)
print('Other data files (not treated as workbook observations):')
for p in sorted((ROOT/'data').iterdir()):
    if p not in inputs: print(p.name)


Excel inputs: 15
2018 Annual Tables_Final (1).xlsx
PSTA5_Indicator_Register (1).xlsx
SAS 2019 _Annual report _Tables.xlsx
SAS 2021_Tables.xlsx
SAS 2022_Tables_0.xlsx
SAS 2023_Annual Tables (3).xlsx
SAS-2016 Final Report.xlsx
Tables_2024 Season A (1).xlsx
Tables_2024 Season B.xlsx
Tables_2025 Season A.xlsx
Tables_2025_Season_B_0.xlsx
Tables_2026 Season A.xlsx
Tables_2026 Season B.xlsx
USAS 2017 Tables.xls
USAS 2020 _Final_Tables.xlsx
Other data files (not treated as workbook observations):
AHS2024_Credits.dta
AHS2024_Saving.dta
AHS2024_Section0.dta
AHS2024_Section11_1_CATTLE MILK PRODUCTION AND USE.dta
AHS2024_Section11_2_EGG PRODUCTION.dta
AHS2024_Section11_3_HONEY PRODUCTION.dta
AHS2024_Section1_HOUSEHOLD MEMBERS CHARACTERISTICS.dta
AHS2024_Section2_LAND TENURE.dta
AHS2024_Section3_4_CROP GROWN, SEEDS AND PRODUCTION & AGRICULTURAL INPUTS AND PRACTICES.dta
AHS2024_Section5_FRUITS PRODUCTION.dta
AHS2024_Section6_EXTENSION SERVICES AND AGRICULTURAL PROGRAMMES.dta
AHS2024_Section7_AGRICUL

## 1. Inspect every sheet before interpreting relationships
`source_cells.json` is a sparse audit of every nonempty source cell, including cell references, formulas and cached values. Missing coordinates inside the reported used range remain blank; they are never zero-filled. Reported dimensions can include formatting-only columns. The inventory also records the actual nonempty extent, merged ranges, comments, notes, missing-value tokens and formula-cache problems. Previews are the first five nonempty rows, with coordinates; longer headers remain in the cell audit.


In [2]:
sheets=[]; inventory=[]; audits=[]
def dump_csv(name, rows):
    if not rows: return
    with (OUT/name).open('w', newline='', encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
def year_hits(text): return sorted(set(re.findall(r'(?<![0-9])(?:19|20)[0-9]{2}(?![0-9])',text)))
def refs(items): return ' | '.join(f"{c['cell']}={c['value']}" for c in items)
for p in inputs:
    if p.suffix.lower()=='.xls':
        wb=xlrd.open_workbook(p)
        source=[]
        for ws in wb.sheets():
            cs=[]
            for r in range(ws.nrows):
                for c in range(ws.ncols):
                    v=ws.cell_value(r,c)
                    if v != '': cs.append(dict(cell=f'{openpyxl.utils.get_column_letter(c+1)}{r+1}',row=r+1,col=c+1,value=v,cached_value=v,type=str(ws.cell_type(r,c)),comment=None))
            source.append((ws.name,ws.nrows,ws.ncols,cs,[str(x) for x in ws.merged_cells], 'Legacy XLS: xlrd returns stored results, not formula expressions; formula-cache completeness cannot be audited.'))
    else:
        wb=openpyxl.load_workbook(p,data_only=False,read_only=False)
        cached=openpyxl.load_workbook(p,data_only=True,read_only=False)
        source=[]
        for ws in wb:
            cs=[]
            for (r,c),obj in sorted(ws._cells.items()):
                if obj.value is not None:
                    cs.append(dict(cell=obj.coordinate,row=r,col=c,value=obj.value,cached_value=cached[ws.title].cell(r,c).value,type=obj.data_type,comment=obj.comment.text if obj.comment else None))
            source.append((ws.title,ws.max_row,ws.max_column,cs,[str(x) for x in ws.merged_cells.ranges],''))
    for name,nr,nc,cs,merged,limitation in source:
        byrow=defaultdict(list)
        for c in cs: byrow[c['row']].append(c)
        preview='\n'.join(refs(byrow[r]) for r in sorted(byrow)[:5])
        top=[c for c in cs if c['row']<=8 and isinstance(c['value'],str)]
        # Titles often use formulas referencing a contents sheet: retain both formula and cached label.
        titles=[str(c['cached_value'] if c['type']=='f' and c['cached_value'] is not None else c['value']) for c in top if re.search(r'table|yield|production|summary|indicator|fertili|seed',str(c['cached_value'] or c['value']),re.I)]
        title=' | '.join(titles[:6])
        header=refs(top)
        notes=[c for c in cs if isinstance(c['value'],str) and re.search(r'(^\s*(source|note|footnote)\b|NISR|National Institute|\*\s*\w)',c['value'],re.I)]
        units=[c for c in cs if isinstance(c['value'],str) and re.search(r'(kg\s*/\s*ha|mt/ha|ton|percentage|percent|\(ha\)|kg/|sqm|rwf|\(%\))',c['value'],re.I)]
        markers=Counter(str(c['value']) for c in cs if isinstance(c['value'],str) and c['value'].strip().lower() in ('-','--','…','...','n/a','na','nan','nil','..',':','(na)','(n/a)'))
        years=year_hits(header+' | '+title)
        filename_years=year_hits(p.name)
        seasons=sorted(set(re.findall(r'\bSeason\s*([ABC])\b',header,re.I)))
        formula=[c for c in cs if c['type']=='f']
        missing_cache=sum(c['cached_value'] is None for c in formula)
        external=sum('[' in str(c['value']) for c in formula)
        row=dict(source_workbook=p.name,source_sheet_or_table=name,reported_rows=nr,reported_columns=nc,nonempty_rows=max((c['row'] for c in cs),default=0),nonempty_columns=max((c['col'] for c in cs),default=0),nonempty_cells=len(cs),title_or_measure_labels=title,header_years='; '.join(years),filename_years='; '.join(filename_years),season_labels='; '.join(seasons),unit_evidence=refs(units[:30]),geography_evidence=refs([c for c in top if re.search(r'district|province|rwanda|national|total',str(c['value']),re.I)]),group_and_denominator_evidence=refs([c for c in top if re.search(r'farmer|plot|land|SSF|LSF|crop|segment|site',str(c['value']),re.I)]),crop_label_evidence=refs([c for c in cs if isinstance(c['value'],str) and re.fullmatch(r'\s*(Maize|Paddy rice|Wheat|Cassava|Sweet potatoes?|Irish potatoes?|Beans|Bush beans?|Climbing beans?|Sorghum|Bananas|Cereals|Legumes.*|Tubers.*)\s*',c['value'],re.I)]),source_notes=refs(notes),missing_tokens=json.dumps(markers),formula_cells=len(formula),formula_without_cached_result=missing_cache,external_formula_cells=external,merged_ranges='; '.join(merged),reader_limitation=limitation,preview=preview)
        inventory.append(row)
        sheet=dict(workbook=p.name,sheet=name,cells=cs,header=header,title=title,years=years,filename_years=filename_years,seasons=seasons,inventory=row)
        sheets.append(sheet)
        audits.append(dict(source_workbook=p.name,sheet=name,reported_dimensions=[nr,nc],merged_ranges=merged,cells=cs,reader_limitation=limitation))
(OUT/'source_cells.json').write_text(json.dumps(audits,ensure_ascii=False,default=str),encoding='utf-8')
dump_csv('sheet_inventory.csv', inventory)
print('Sheets inspected:',len(inventory))
for p in inputs:
    ss=[s for s in inventory if s['source_workbook']==p.name]
    print(p.name, '—',len(ss),'sheets; header year labels:',sorted(set(y for s in ss for y in s['header_years'].split('; ') if y)))
print('Sheets with missing formula caches:',sum(s['formula_without_cached_result']>0 for s in inventory))
print('Sheets with external formulas:',sum(s['external_formula_cells']>0 for s in inventory))


Sheets inspected: 871
2018 Annual Tables_Final (1).xlsx — 55 sheets; header year labels: ['2017', '2018']
PSTA5_Indicator_Register (1).xlsx — 5 sheets; header year labels: ['2023', '2024', '2025', '2026']
SAS 2019 _Annual report _Tables.xlsx — 71 sheets; header year labels: ['2017', '2018', '2019']
SAS 2021_Tables.xlsx — 67 sheets; header year labels: ['2019', '2020', '2021']
SAS 2022_Tables_0.xlsx — 68 sheets; header year labels: ['2020', '2021', '2022']
SAS 2023_Annual Tables (3).xlsx — 79 sheets; header year labels: ['2017', '2022', '2023']
SAS-2016 Final Report.xlsx — 129 sheets; header year labels: ['2015', '2016']
Tables_2024 Season A (1).xlsx — 34 sheets; header year labels: ['2017', '2023', '2024']
Tables_2024 Season B.xlsx — 33 sheets; header year labels: ['2023', '2024']
Tables_2025 Season A.xlsx — 34 sheets; header year labels: ['2017', '2024', '2025']
Tables_2025_Season_B_0.xlsx — 33 sheets; header year labels: ['2023', '2024', '2025']
Tables_2026 Season A.xlsx — 35 sheets;

## 2. Inspect register definitions and periods independently
All original baseline and target strings are retained in `register_fields_for_review.csv`. The register's own seasonal periods take precedence over assumptions about fiscal years. Its policy status and target provenance are unresolved: the supplied register is not an authenticated PSTA5 results framework. No external policy source has been used to validate it.


In [3]:
register=[]; indicators={}
for s in sheets:
    if 'PSTA5' not in s['workbook']: continue
    grid={(c['row'],c['col']):c['value'] for c in s['cells']}
    for r in range(2,s['inventory']['reported_rows']+1):
        name=grid.get((r,1))
        if not name: continue
        if name not in indicators:
            indicators[name]=dict(indicator_id=f'REG-{len(indicators)+1:02}',psta5_indicator=name,definition=grid.get((r,2)),unit=grid.get((r,3)),crop_or_group=grid.get((r,7)))
        register.append(dict(**indicators[name],register_sheet=s['sheet'],source_row=r,baseline_field=grid.get((1,4)),baseline_value_as_written=grid.get((r,4)),target_field=grid.get((1,5)),target_value_as_written=grid.get((r,5)),stated_source=grid.get((r,6)),policy_provenance='UNRESOLVED: supplied register field labels retained; not independently verified PSTA5 targets',geography='Not explicitly stated in register'))
dump_csv('register_fields_for_review.csv',register)
for x in indicators.values(): print(x)
print('Register records:',len(register))


{'indicator_id': 'REG-01', 'psta5_indicator': 'Quantity of production by crop', 'definition': 'Stated volume of crop production for main categories', 'unit': 'Metric Tons (MT)', 'crop_or_group': 'Cereals, Tubers/Roots, Bananas, Legumes/Pulses'}
{'indicator_id': 'REG-02', 'psta5_indicator': 'Productivity for priority food crops', 'definition': 'Average productivity measured by yield per harvested area', 'unit': 'Metric Tons per Hectare (MT/ha)', 'crop_or_group': 'Maize, Paddy Rice, Wheat, Cassava, Sweet potato, Irish potato, Beans'}
{'indicator_id': 'REG-03', 'psta5_indicator': 'Small-scale farmers using improved seeds', 'definition': 'Percentage of small-scale farmers who applied improved seeds', 'unit': 'Percentage (%)', 'crop_or_group': 'Small-Scale Farmers (SSF)'}
{'indicator_id': 'REG-04', 'psta5_indicator': 'Large-scale farmers using improved seeds', 'definition': 'Percentage of large-scale farmers who applied improved seeds', 'unit': 'Percentage (%)', 'crop_or_group': 'Large-Scal

## 3. Candidate table/year relationships
Candidate inclusion is deliberately broad: sheets containing production, yield/productivity, harvested/cultivated area, improved seeds, inorganic fertilizer, or summary main indicators in their initial labels. Each of the six distinct register definitions is considered against every included table/year; unrelated topic pairs are contextual. Other sheets remain visible in the inventory and exclusion audit. A year mentioned only in a filename is provisional, and a year in a multiyear header does not establish the reference period of every cell. No observation values are extracted into this crosswalk.

POSSIBLE MATCH requires direct concept evidence and, for input use, explicit farmer denominator and requested farmer type. Plot/land measures and seed/fertilizer composition or sources are context. Missing denominator/group evidence is UNRESOLVED. Yield labels remain provisional pending crop and harvested-area denominator verification. Mixed summary tables are UNRESOLVED until their individual blocks are mapped. The exact source strings and coordinate evidence accompany every row; analytical topic labels do not replace source categories.


In [4]:
crosswalk=[]; excluded=[]
for s in sheets:
    if 'PSTA5' in s['workbook']: continue
    h=(s['header']+' | '+s['title']).lower(); title=s['title'].lower()
    families=[]
    if re.search(r'production|produced',h): families.append('production')
    if re.search(r'yield|productivity',h): families.append('yield')
    if re.search(r'harvested area|cultivated area',h): families.append('area')
    if re.search(r'improved seed|type of seeds|types of seeds|seed use|use of seeds|seeds used',h): families.append('seed')
    if re.search(r'inorganic fertili',h): families.append('fertilizer')
    summary=bool(re.search(r'summary.*main indicator|main sas indicator|sas main indicator',h))
    if summary: families.append('summary')
    if not families or re.search(r'list of table',s['sheet'],re.I) or s['sheet'].lower()=='summary':
        excluded.append(dict(source_workbook=s['workbook'],source_sheet=s['sheet'],reason='Contents/navigation or no candidate concept in first eight rows; inspect inventory for scope extension',exact_header=s['header']))
        continue
    yearlist=s['years'] or s['filename_years'] or ['UNRESOLVED']
    for ind in indicators.values():
        name=ind['psta5_indicator']; target_topic='production' if name.startswith('Quantity') else 'yield' if name.startswith('Productivity') else 'seed' if 'seeds' in name else 'fertilizer'
        group='SSF' if name.startswith('Small') else 'LSF' if name.startswith('Large') else ''
        status='CONTEXT ONLY'; reason='Different measured concept; potential background or supporting input only.'
        if summary:
            status='UNRESOLVED'; reason='Mixed main-indicator summary: map exact row, year/season columns, unit and population before determining relationship.'
        elif target_topic=='production' and re.search(r'use of (crop )?production|production use|share of crop production|sampling error',title):
            reason='Production allocation/use shares or sampling-error statistics do not directly measure crop production volume.'
        elif target_topic in families:
            if target_topic in ('production','yield'):
                status='POSSIBLE MATCH'; reason='Direct production/yield concept, subject to exact crop, geography, population, unit and period checks.'
            else:
                subtype=bool(re.search(r'source of|sources of|type of inorganic|types of inorganic|source.*seed',title))
                farmer=bool(re.search(r'percentage of farmers|farmers who|farmer type|farmers.*us|farmers.*appl',h))
                hasgroup=bool(re.search(r'\b'+group+r'\b',h,re.I) or re.search('small.?scale' if group=='SSF' else 'large.?scale',h,re.I))
                if subtype:
                    reason='Source/composition of input use does not measure farmer adoption prevalence.'
                elif farmer and hasgroup:
                    status='POSSIBLE MATCH'; reason='Farmer-use concept and requested group appear in headers; select farmer block only, never plot or land blocks.'
                elif 'plot' in title and not farmer:
                    reason='Plot denominator differs from the register farmer denominator.'
                else:
                    status='UNRESOLVED'; reason='Input-use topic present, but farmer denominator and/or requested farmer group is not explicit enough in table header.'
        elif target_topic=='yield' and ('production' in families or 'area' in families):
            reason='Supporting numerator/area context only; do not derive yield without matched crop, harvested-area denominator and documented estimator.'
        for year in yearlist:
            note=reason+' Period alignment with each register season must be checked; no annual aggregation. Register policy target provenance remains unresolved.'
            if not s['years']:
                note+=' Year inferred from filename only, not verified in header.'
                if status=='POSSIBLE MATCH': status='UNRESOLVED'
            if s['inventory']['formula_without_cached_result']:
                note+=' Missing formula caches: do not treat blank cached results as observed values.'
            conversion='No conversion applied; verify units in exact block.'
            if target_topic=='yield': conversion='If source is kg/ha and register MT/ha, divide by 1000 only after denominator/crop verification; no conversion applied.'
            crosswalk.append(dict(indicator_id=ind['indicator_id'],psta5_indicator=name,psta5_definition=ind['definition'],psta5_unit=ind['unit'],psta5_crop_or_group=ind['crop_or_group'],register_records='; '.join(r['register_sheet']+' row '+str(r['source_row']) for r in register if r['indicator_id']==ind['indicator_id']),source_workbook=s['workbook'],source_sheet_or_table=s['sheet'],year=year,year_basis='header evidence (may be comparative; verify column)' if s['years'] else 'filename only — unresolved',season='; '.join(s['seasons']) or 'UNRESOLVED',nisr_source_label=s['title'],exact_header_evidence=s['header'],definition_and_calculation='Not established from table label; inspect source cells/formulas and survey methodology. No calculation performed.',unit_evidence=s['inventory']['unit_evidence'] or 'Not explicit in inspected labels',crop_or_group_evidence=s['inventory']['group_and_denominator_evidence'] or 'UNRESOLVED; see source cells',exact_crop_label_evidence=s['inventory']['crop_label_evidence'],geography_evidence=s['inventory']['geography_evidence'] or 'UNRESOLVED',denominator='Keep farmer, plot, land and harvested-area denominators separate; exact block verification required',source_note=s['inventory']['source_notes'] or 'No explicit source note detected; local workbook provenance only',missing_or_not_applicable=s['inventory']['missing_tokens']+'; blanks preserved; token meanings unverified',unit_conversion_required=conversion,value_role='NISR table candidate; no observed value extracted; register baselines/targets are separate',match_status=status,comparability_note=note))
dump_csv('candidate_crosswalk.csv',crosswalk)
dump_csv('noncandidate_sheets.csv',excluded)
print('Candidate rows:',len(crosswalk)); print(dict(Counter(r['match_status'] for r in crosswalk)))
for ind in indicators.values():
    rr=[r for r in crosswalk if r['indicator_id']==ind['indicator_id']]
    print(ind['indicator_id'],ind['psta5_indicator'],dict(Counter(r['match_status'] for r in rr)))
assert len({(r['indicator_id'],r['source_workbook'],r['source_sheet_or_table'],r['year']) for r in crosswalk})==len(crosswalk)
assert {r['match_status'] for r in crosswalk} <= {'POSSIBLE MATCH','CONTEXT ONLY','UNRESOLVED'}
assert set(ind['indicator_id'] for ind in indicators.values()) == set(r['indicator_id'] for r in crosswalk)


Candidate rows: 2520
{'UNRESOLVED': 204, 'CONTEXT ONLY': 2126, 'POSSIBLE MATCH': 190}
REG-01 Quantity of production by crop {'UNRESOLVED': 18, 'CONTEXT ONLY': 352, 'POSSIBLE MATCH': 50}
REG-02 Productivity for priority food crops {'UNRESOLVED': 18, 'CONTEXT ONLY': 334, 'POSSIBLE MATCH': 68}
REG-03 Small-scale farmers using improved seeds {'UNRESOLVED': 43, 'CONTEXT ONLY': 357, 'POSSIBLE MATCH': 20}
REG-04 Large-scale farmers using improved seeds {'UNRESOLVED': 47, 'CONTEXT ONLY': 357, 'POSSIBLE MATCH': 16}
REG-05 Small-scale farmers using inorganic fertilizer {'UNRESOLVED': 37, 'CONTEXT ONLY': 363, 'POSSIBLE MATCH': 20}
REG-06 Large-scale farmers using inorganic fertilizer {'UNRESOLVED': 41, 'CONTEXT ONLY': 363, 'POSSIBLE MATCH': 16}


## 4. Readable review artifacts and preservation checks
The HTML report displays the entire inventory and candidate crosswalk, with search filtering and source previews. No verified tidy observation CSV is created at this review gate. CSVs here contain metadata, source register fields and candidate relationships only.


In [5]:
def table(rows, fields):
    return '<table><thead><tr>'+''.join('<th>'+html.escape(k)+'</th>' for k in fields)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+html.escape(str(r.get(k,'')))+'</td>' for k in fields)+'</tr>' for r in rows)+'</tbody></table>'
style='<style>body{font:15px system-ui;margin:30px;color:#20332d}table{border-collapse:collapse;width:100%;margin:20px 0}td,th{border:1px solid #ccc;padding:8px;text-align:left;vertical-align:top;white-space:pre-wrap;min-width:100px}th{background:#e6eee8}section{overflow:auto}input{padding:10px;width:65%}h2{margin-top:35px}</style>'
intro=f'<h1>APASA workbook review — provisional</h1><p>{len(inputs)} Excel files; {len(inventory)} sheets; {len(indicators)} distinct register definitions; {len(register)} seasonal register records; {len(crosswalk)} candidate table/year relationships.</p><p>Review gate: no cleaned observations exported. POSSIBLE MATCH is not verified comparability. Register target provenance remains unresolved. Years in comparative headers require column-level review.</p><p>Search each table by indicator, workbook, sheet or year. Full coordinate evidence is in the CSV and source_cells.json.</p>'
report=style+intro+'<p><a href="review_note.md">Read the unresolved issues and source-cell examples</a></p>'
for label,rows,fields in [
    ('Inventory',inventory,['source_workbook','source_sheet_or_table','reported_rows','reported_columns','nonempty_rows','nonempty_columns','title_or_measure_labels','header_years','filename_years','season_labels','preview','source_notes','missing_tokens','formula_without_cached_result','external_formula_cells','reader_limitation']),
    ('Register definitions and separately retained baseline/target fields',register,list(register[0])),
    ('Candidate crosswalk',crosswalk,['indicator_id','psta5_indicator','source_workbook','source_sheet_or_table','year','year_basis','season','nisr_source_label','match_status','comparability_note','unit_conversion_required','crop_or_group_evidence'])]:
    report+='<h2>'+label+'</h2><input placeholder="Filter this table"><section>'+table(rows,fields)+'</section>'
report+="<script>document.querySelectorAll('input').forEach(el=>el.addEventListener('input',()=>{for(const r of el.nextElementSibling.querySelectorAll('tbody tr'))r.hidden=!r.textContent.toLowerCase().includes(el.value.toLowerCase())}));</script>"
(OUT/'review.html').write_text('<!doctype html><meta charset="utf-8"><title>APASA workbook review</title>'+report,encoding='utf-8')
after={str(p.relative_to(ROOT)):hash_file(p) for p in inputs}
assert before==after, 'Input changed during inspection'
(OUT/'input_manifest.json').write_text(json.dumps({'before':before,'after':after,'unchanged':before==after},indent=2))
print('Verified all input SHA-256 hashes unchanged. Open review.html for the full inventory and crosswalk.')


Verified all input SHA-256 hashes unchanged. Open review.html for the full inventory and crosswalk.


# APASA interpretation review — awaiting review before cleaned exports

The directory contains **15 Excel workbooks: one register and 14 NISR workbooks**, covering 871 sheets in total. There is no single assembled 2013-onward workbook in `data/`. NISR filenames span 2016–2026; some headers include 2015 comparison data. No 2013 or 2014 source workbook was found. A header year is evidence of a label, not proof of a complete observation series. `local_data/references/day7/Tables_2026_Season_B.xlsx` is an additional reference copy outside the requested data directory and was not substituted for the supplied input.

The register is `PSTA5_Indicator_Register (1).xlsx`: five seasonal sheets, each 7 rows × 7 columns, containing six recurring indicator definitions (30 seasonal records). There are no official indicator codes, no explicit geography field, no detailed estimator specifications, no authenticated policy-framework citation, and no fiscal-year targets in this supplied register. Local REG-01–REG-06 IDs identify the six definitions; every candidate links to all five original seasonal register records. Baselines refer to the corresponding prior-year season; targets refer to the sheet's season. These seasonal records have not been collapsed into annual targets.

## Most important unresolved issue: what are the register's targets?

The register cites NISR SAS as its stated source. Its 2024 B target values reproduce NISR observations in `Tables_2024 Season B.xlsx`:

| Register field | NISR source cell | NISR stored result |
|---|---|---|
| Maize production: 119,101.23 MT | Table 5!G6 | 119101.22948988932 MT |
| Maize productivity: 1.284 MT/ha | Table 5!I6 | 1.2841270272071035 MT/ha |
| SSF improved seeds: 16.64% | Table 17!D36 | 16.64 |
| LSF improved seeds: 72.57% | Table 17!E36 | 72.57 |
| SSF inorganic fertilizer: 51.20% | Table 22!D35 | 51.2 |
| LSF inorganic fertilizer: 82.89% | Table 22!E35 | 82.89 |

The first two coincide after rounding; the other four coincide numerically. This does not establish how the register was compiled, nor authenticate a policy target. Preserve the register field labels but request the official PSTA5 results framework, indicator IDs, baseline methodology, target approval/source and target periods before using these as policy targets. No target-achievement inference is justified.

## Comparability checks still needed

- Crop production: distinguish volume (MT) from shares of production use, uncertainty estimates and category aggregates. The register names broad crop categories but gives only maize baseline/target examples. Do not extend maize values to other crops or sum crop totals and their component crops.
- Productivity: match exact crop and farmer population, and verify harvested area rather than cultivated area. Table 5 explicitly distinguishes both area columns. A source yield in kg/ha could be converted to MT/ha by dividing by 1000, but no conversion was performed. Survey estimators and treatment of mixed cropping still require methodology review.
- Improved seeds and inorganic fertilizer: select the percentage-of-farmers block and the correct SSF or LSF column. Farmer, plot and land percentages coexist in the same sheets. Seed/fertilizer composition or source shares do not measure adoption prevalence. Group definitions, thresholds, weights and survey coverage may change across years.
- Periods: seasonal A/B/C observations remain separate. Comparative years inside one table do not mean all measures cover all years. Header-year relationships are provisional until the exact columns are checked; filename-only years are UNRESOLVED. Do not assume a year associated with constant prices is the observation year. No annual totals or fiscal-year alignment was calculated.
- Geography: national, province and district rows must be selected explicitly; do not add national totals to component districts. Register geography is unstated. Local file names do not authenticate publication provenance.
- Missing values: literal strings including `N/A`, `(NA)`, dashes and blanks have different possible meanings. No codebook meanings have been assumed. Formula results may also be empty strings, zero or missing cached values; these are distinct.
- Five sheets contain formulas without cached results: 2018 Table 10.1 (4), 2018 Appendix 9 (86), SAS 2021 Table 40 (1), USAS 2020 Table 35 (1), and USAS 2020 Table 38 (1). There are external formulas in 23 sheets. Formula expressions and cached results are preserved separately; external references were not refreshed. The legacy XLS reader exposes stored results but cannot provide the same formula audit.

## How to review these outputs

Open `review.html` for the full searchable inventory, register fields and candidate crosswalk. `sheet_inventory.csv` describes every sheet and previews its structure. `source_cells.json` retains every nonempty source cell, coordinates, formulas, cached results, comments and merged ranges; absent coordinates remain blank. `noncandidate_sheets.csv` documents sheets outside the initial topic screen. The candidate screen is based on the first eight rows and is intentionally provisional; tables with unexpected header placement can require scope extension. Full-cell data is preserved so those exclusions are reviewable.

The crosswalk has one status per distinct register indicator × candidate workbook/sheet × header year (or unresolved filename year). It is a broad table-level candidate map, not a verified observation-level crosswalk. Exact crop rows, season/year columns, denominator blocks and survey methodology must be checked before extraction. Different register seasonal occurrences are linked through `register_records`; candidate historical seasons are not assumed aligned to every register period. Source labels are preserved alongside analytical IDs and interpretations.

All input SHA-256 hashes are verified unchanged in `input_manifest.json`. No workbook was saved, no cleaned observations were exported, and baseline/target fields remain in `register_fields_for_review.csv`. Review the target provenance, crop scope, farmer denominators and year/season mapping before authorizing the next extraction stage.
